In [ ]:
import os 
import pandas as pd 
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from imblearn.over_sampling import SMOTE
from sklearn.decomposition import TruncatedSVD
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
import os
import pickle
import mlflow
import dagshub
from flaml import AutoML
from sklearn.metrics import accuracy_score

# Path Configuration:
data_path = os.path.join("data", "churn.csv")  
model_dir = os.path.join("models")            
os.makedirs(model_dir, exist_ok=True)
pickle_path = os.path.join(model_dir, "churn_model.pkl")

# Data Loading 
def data_ingestion():    
    # Dataloading and it will return Dataframe
    df = pd.read_csv(data_path)
    return df

## Data Preprocessing

def data_preprocessing(df):

    df = df.drop_duplicates()

    df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

    X = df.drop(columns=['Churn', 'customerID'], errors='ignore')
    y = df['Churn']

    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=0.3,
        random_state = 1
    )

    numerical_col = X_train.select_dtypes(exclude='object').columns
    categorical_col = X_train.select_dtypes(include='object').columns

    numerical_pipeline = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', MinMaxScaler())
    ])

    categorical_pipeline = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OneHotEncoder(handle_unknown='ignore'))
    ])

    preprocessor = ColumnTransformer(transformers=[
        ('num', numerical_pipeline, numerical_col),
        ('cat', categorical_pipeline, categorical_col)
    ])

    X_train = preprocessor.fit_transform(X_train)
    X_test = preprocessor.transform(X_test)

    sm = SMOTE(sampling_strategy=0.5, random_state=1)
    X_train, y_train = sm.fit_resample(X_train, y_train)

    # Step 1: Fit SVD with higher components
    svd_temp = TruncatedSVD(n_components=100, random_state=1)
    X_train_temp = svd_temp.fit_transform(X_train)

    # Step 2: Calculate cumulative variance
    cumulative_variance = np.cumsum(svd_temp.explained_variance_ratio_)

    # Step 3: Find components for 95% variance
    n_components_95 = np.argmax(cumulative_variance >= 0.95) + 1

    # Step 4: Final SVD
    svd = TruncatedSVD(n_components=n_components_95, random_state=1)
    X_train = svd.fit_transform(X_train)
    X_test = svd.transform(X_test)

    print(f"Selected Components for 95% variance: {n_components_95}")

    return X_train, X_test, y_train, y_test, preprocessor,svd

# Model Building

def model_building(X_train, X_test, y_train, y_test, preprocessor, svd):

    dagshub.init(
        repo_owner='chandanc5525',
        repo_name='CustomerChurn_PredictionModel',
        mlflow=True
    )

    mlflow.set_experiment("Customer_Churn_AutoML")

    with mlflow.start_run():

        automl = AutoML()

        settings = {
            "time_budget": 60,
            "metric": "accuracy",
            "task": "classification",
            "estimator_list":  ["rf", "lgbm", "xgboost", "extra_tree", "lrl2", "svc"]
        }

        automl.fit(X_train=X_train, y_train=y_train, **settings)

        if automl.model is None:
            raise Exception("AutoML failed")

        y_pred = automl.predict(X_test)
        acc = accuracy_score(y_test, y_pred)

        print("Best Model:", automl.model)
        print("Accuracy:", acc)

        mlflow.log_metric("accuracy", acc)
        mlflow.log_param("best_model", type(automl.model).__name__)
        mlflow.sklearn.log_model(automl.model, "model")

        # Saving Pickle File for Model Deployment 
        os.makedirs("models", exist_ok=True)

        file_path = os.path.join("models", "churn_model.pkl")

        model_package = {
            "model": automl.model,
            "preprocessor": preprocessor,
            "svd": svd
        }

        with open(file_path, "wb") as f:
            pickle.dump(model_package, f)

        print("Model saved at:", file_path)
    

        return automl.model

In [ ]:
def main():
    
    # Step1: Data Ingestion
    df = data_ingestion()
    print(df.shape)
    
    # Step2: Data Preprocessing
    X_train,X_test,y_train,y_test,preprocessor,svd = data_preprocessing(df)
    print(X_train.shape)
    print(X_test.shape)
    
    # Step3: Model Building
    
    model = model_building(X_train, X_test, y_train, y_test,preprocessor,svd)

    print(model)
    
    
main()